# Pilot on real data with a free scorer (CPU, about 15 to 25 minutes)

Runs the whole twin pipeline on the **pilot slice** (12.5% of authors, chosen by their salted hash) of Zenodo 4266616, scored with VADER negative affect. It proves your environment works and gives design evidence. It is NOT a paper result and does not use StressTwin-F.

**Kaggle:** Add Input = your private dataset `stresstwin-p`; Settings: Accelerator None, **Internet On**; then *Save & Run All*.
**Colab:** upload the unzipped package folder to Drive (or run `!unzip` on the zip), set `PKG` in cell 1.

Raw text stays inside this notebook's working folder. Do not publish its output folder; the aggregate files `report.md` and `results.json` are safe to share.

In [ ]:
import os, sys, subprocess, shutil, hashlib, glob, secrets
ON_KAGGLE = os.path.exists('/kaggle')
WORK = '/kaggle/working/work' if ON_KAGGLE else '/content/work'
os.makedirs(WORK, exist_ok=True)
LOCAL_FOLDER = None            # set to a folder holding the 12 .pkl files to skip the download (e.g. data you uploaded yourself)
PKG = None                     # Colab only: folder that contains pyproject.toml
if ON_KAGGLE:
    def find_pkg():
        """Find the package wherever Kaggle mounted the dataset (the mount path differs between Kaggle versions), or unzip it if it was left zipped."""
        import zipfile
        for r, dirs, files in os.walk('/kaggle/input'):
            if 'pyproject.toml' in files and os.path.isdir(os.path.join(r, 'stresstwin_p')):
                return r
        for r, dirs, files in os.walk('/kaggle/input'):
            for f in files:
                if f.endswith('.zip') and 'stresstwin' in f.lower() and 'reproducibility' not in f.lower():
                    out = '/kaggle/working/pkg_unzipped'; zipfile.ZipFile(os.path.join(r, f)).extractall(out)
                    for r2, d2, f2 in os.walk(out):
                        if 'pyproject.toml' in f2 and os.path.isdir(os.path.join(r2, 'stresstwin_p')):
                            return r2
        print('Nothing found. What is mounted under /kaggle/input:')
        for r, d, f in os.walk('/kaggle/input'):
            if r.count(os.sep) <= 5: print(' ', r, '| files:', f[:6])
        return None
    root = find_pkg()
    assert root, 'package not found: in the right panel click Add Input -> Your Datasets -> stresstwin-p (it must be listed under Input), then run again'
else:
    root = None
    for r, dirs, files in os.walk(PKG or '/content/drive/MyDrive/stresstwin_p'):
        if 'pyproject.toml' in files and os.path.isdir(os.path.join(r, 'stresstwin_p')):
            root = r; break
    assert root, 'package not found: set PKG to the folder that contains pyproject.toml'
dst = f'{WORK}/pkg'
if not os.path.exists(dst): shutil.copytree(root, dst)
def sh(*cmd, **kw):
    r = subprocess.run(list(cmd), check=True, **kw); return r
sh(sys.executable, '-m', 'pip', '-q', 'install', '-e', dst, 'vaderSentiment', 'openpyxl')
sys.path.insert(0, dst)          # the running kernel does not see a package installed after it started
print('package ready at', dst)

In [ ]:
# 2) data: download Zenodo 4266616 (CC BY 4.0), check md5, unzip. Skipped when LOCAL_FOLDER is set.
RAW = LOCAL_FOLDER
if RAW is None:
    RAW = f'{WORK}/raw'; os.makedirs(RAW, exist_ok=True)
    zp = f'{RAW}/MentalHealthSubreddits2017.zip'
    if not os.path.exists(zp):
        sh('wget', '-q', '-O', zp, 'https://zenodo.org/records/4266616/files/MentalHealthSubreddits2017.zip?download=1')
    h = hashlib.md5(open(zp, 'rb').read()).hexdigest()
    assert h == '64a61e5883f2dbe80fbfdf173df984f9', f'md5 mismatch {h}: do not continue'
    sh('unzip', '-q', '-o', zp, '-d', RAW)
pk = sorted(p for p in glob.glob(f'{RAW}/**/*.pkl', recursive=True) if '__MACOSX' not in p and not os.path.basename(p).startswith('._'))
print(len(pk), 'pickles (expect 12):'); [print(' ', os.path.basename(p)) for p in pk]

In [ ]:
# 3) secret salt. It decides which authors are in the pilot slice, so it must be THE SAME in every run.
#    Kaggle: Add-ons -> Secrets -> Add Secret, label STRESSTWIN_SALT, value = any long random phrase. Keep a copy in a password manager (never in a shared notebook).
sf = f'{WORK}/salt.bin'
try:
    from kaggle_secrets import UserSecretsClient
    open(sf, 'wb').write(hashlib.sha256(UserSecretsClient().get_secret('STRESSTWIN_SALT').encode()).digest())
    print('salt taken from the Kaggle secret STRESSTWIN_SALT (reproducible across runs)')
except Exception as e:
    if not os.path.exists(sf): open(sf, 'wb').write(secrets.token_bytes(32))
    print('WARNING: no STRESSTWIN_SALT secret found; using a random salt. Results are NOT reproducible and the pilot slice will change on every run.', repr(e)[:80])
sh(sys.executable, '-m', 'stresstwin_p.timeline_builder', '--folder', RAW, '--salt-file', sf, '--out', f'{WORK}/tables')
# keep the output lean and ready to turn into the private dataset `stresstwin-tables`
shutil.copytree(f'{WORK}/tables', '/kaggle/working/stresstwin_tables' if ON_KAGGLE else f'{WORK}/stresstwin_tables', dirs_exist_ok=True)
if RAW.startswith(WORK): shutil.rmtree(RAW, ignore_errors=True)       # the downloaded zip and pickles are not needed again

In [ ]:
# 4) pilot slice = authors whose hash starts with 0 or 1; score with VADER (also the two halves, for split-half noise)
import pandas as pd
from stresstwin_p import score_texts as ST
from stresstwin_p.lexicon_scorer import vader_scorers
T = f'{WORK}/tables/'
o = pd.read_csv(T + 'observations.csv'); thr = pd.read_csv(T + 'threads.csv'); tx = pd.read_csv(T + 'texts.csv')
pil = o.author.astype(str).str[0].isin(list('01'))
o = o[pil]; thr = thr[thr.author.astype(str).str[0].isin(list('01'))]; tx = tx[tx.obs_id.isin(o.obs_id)]
print('pilot observations', len(o), 'authors', o.author.nunique())
sc = ST.score_table(tx, vader_scorers())
o.drop(columns=['n_chars']).merge(sc[['obs_id', 'n_chars']], on='obs_id').to_csv(f'{WORK}/pilot_obs.csv', index=False)
sc.to_csv(f'{WORK}/pilot_scores.csv', index=False); thr.to_csv(f'{WORK}/pilot_threads.csv', index=False)

In [ ]:
# 5) the twin battery (about 5 to 10 minutes): baselines, noise arms, dynamics/pooling ablations, intervals, reply association
sh(sys.executable, '-m', 'stresstwin_p.run_all', '--obs', f'{WORK}/pilot_obs.csv', '--scores', f'{WORK}/pilot_scores.csv',
   '--threads', f'{WORK}/pilot_threads.csv', '--out', f'{WORK}/pilot_out', '--seed', '0')

In [ ]:
# 6) read the headline numbers: twin minus each baseline, NLL (negative = twin better)
import json
r = json.load(open(f'{WORK}/pilot_out/results.json'))
print('eligible authors', r['eligible_authors'], '| skipped', r.get('skipped'))
for c, x in r['communities'].items():
    w = x['within']['summary']
    print(c, {m: round(w[m]['nll'], 3) for m in ('author_mean', 'global_mean', 'shortcut', 'twin[split_half]', 'twin[split_half/static]') if m in w})
print(open(f'{WORK}/pilot_out/report.md').read()[:3000])